# RE_103 · Com de greu va ser la sequera

**Optativa d'Aprenentatge automàtic · Bloc 1 — Dades · Dificultat 2**

> **Repte resolt.** Una manera de resoldre'l, no l'única. Els números del text surten d'una execució real del quadern.

Ets tècnic o tècnica de l'Agència Catalana de l'Aigua. Després de la sequera del 2022-2024, el
Govern vol un informe curt amb dades de tots els embassaments de les conques internes des de
l'any 2000.

**La pregunta:** com de greu va ser la sequera comparada amb les anteriors, i com estan avui les
reserves? A quin moment del passat s'assemblen més?

L'enunciat complet, amb el diccionari de columnes, és a [RE_103](../RE_103_embassaments.ipynb). Si encara no l'has fet, fes-lo abans de llegir això.

In [ ]:
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

FITXER = "embassaments_conques_internes.csv"
URL = "https://raw.githubusercontent.com/ZenidX/Machine-Learning/main/Machine%20Learning/04_reptes/dades/" + FITXER
ruta = URL
for candidata in (os.path.join("dades", FITXER), os.path.join("..", "dades", FITXER)):
    if os.path.exists(candidata):
        ruta = candidata
df = pd.read_csv(ruta)
print(ruta, df.shape)

## Nivell 1 · Explorar

### 1.1 · Forma, temps i embassaments

Quantes files i columnes hi ha? Converteix `dia` a data. Quants embassaments hi ha, i quina és la primera i l'última data? Quantes files té cada embassament?

In [ ]:
print(df.shape)
df["dia"] = pd.to_datetime(df["dia"])
print(df["dia"].dtype, df["dia"].min(), df["dia"].max())
print(df["estaci"].nunique())
print(df["estaci"].value_counts())

### 1.2 · Nuls i repetides

Quants buits hi ha per columna? Hi ha cap parella dia-embassament repetida? Si en falta alguna, quantes n'hi haurien d'haver?

In [ ]:
print(df.isna().sum())
print("repetides:", df.duplicated(subset=["dia", "estaci"]).sum())
print("esperades:", df["dia"].nunique() * df["estaci"].nunique(), "· files:", len(df))
df[df["percentatge_volum_embassat"].isna()]

### 1.3 · Valors amb sentit físic

Mira el resum numèric de les tres columnes de mesura. Hi ha valors que no tinguin sentit físic? Si en trobes, compta'ls per embassament.

In [ ]:
print(df.describe().round(1))
fora = df["percentatge_volum_embassat"] > 100
print("total:", fora.sum())
print(fora.groupby(df["estaci"]).sum().sort_values(ascending=False))

**Comentari.** 3.391 lectures per sobre del 100 %: 3.217 del Foix, i la resta de la Llosa del Cavall (124), Riudecanyes (27) i Susqueda (23). El Foix hi és durant anys seguits, de manera que no és un error puntual: la capacitat de referència amb què es calcula el percentatge no és la que l'embassament pot tenir de debò. No sabem el motiu exacte; el que importa és que el percentatge del Foix no és comparable amb el dels altres, i per això la reserva total es fa amb hm³. També hi ha 54 buits a `nivell_absolut` i 1 a `percentatge_volum_embassat` (7-maig-2026).

## Nivell 2 · Respondre

### 2.1 · La corba de cada embassament

Fes una taula ampla amb una fila per dia i una columna per embassament, amb el percentatge de volum. Dibuixa-la: una línia per embassament, del 2000 fins avui. Es veu la sequera?

In [ ]:
pct = df.pivot(index="dia", columns="estaci", values="percentatge_volum_embassat").sort_index()
print(pct.shape, pct.index.is_monotonic_increasing)
pct.plot(figsize=(12, 5), legend=False, title="% de volum de cada embassament")
plt.ylabel("% de capacitat")
plt.show()

### 2.2 · La reserva total

Calcula la reserva total de les conques internes, **en hm³**, per a cada dia, i dibuixa-la.

In [ ]:
total = df.groupby("dia")["volum_embassat"].sum()
print(len(total), df["dia"].nunique())
total.plot(figsize=(12, 4), title="Reserva total de les conques internes (hm³)")
plt.ylabel("hm³")
plt.show()

**Comentari.** Si algú fa la mitjana dels percentatges dels nou embassaments, dona el mateix pes a la Baells que a Riudecanyes, que és molt més petit.

### 2.3 · El pitjor moment

Quin dia la reserva total va ser mínima, i quant valia? Després calcula el **mínim de cada any** i fes-ne un gràfic de barres. Compara la sequera del 2022-2024 amb les anteriors.

In [ ]:
print(total.idxmin(), round(total.min(), 1))
minim_any = total.groupby(total.index.year).min()
print(minim_any.round(1).sort_values().head(6))
minim_any.plot(kind="bar", figsize=(12, 4), title="Reserva mínima de cada any (hm³)")
plt.show()

### 2.4 · Avui contra el passat

Agafa l'últim dia de les dades. Quina reserva total hi ha? Compara-la amb la reserva **el mateix dia i mes** de cada any anterior. Quants anys n'hi havia més?

In [ ]:
ultim = total.index.max()
mateix_dia = total[(total.index.month == ultim.month) & (total.index.day == ultim.day)]
print(ultim.date(), round(total[ultim], 1))
print(mateix_dia.round(1).to_string())
print("anys amb més reserva que avui:", (mateix_dia > total[ultim]).sum(), "de", len(mateix_dia) - 1)

### 2.5 · A quin dia del passat s'assembla avui (NumPy)

Fes servir la taula ampla de 2.1: cada dia és un vector de nou percentatges, un per embassament.

1. Passa-la a un array `P` (decideix abans què fas amb les files que tenen algun buit).
2. Agafa com a referència el vector de l'últim dia.
3. Calcula la distància euclidiana de cada dia a la referència.
4. Quins són els cinc dies **d'abans del 2026** més semblants a avui? De quins anys són?

In [ ]:
pct_net = pct.dropna()
P = pct_net.to_numpy()
print(P.shape)
d = np.linalg.norm(P - P[-1], axis=1)
print("a si mateix:", d[-1], "· NaN:", np.isnan(d).sum())
dist = pd.Series(d, index=pct_net.index)
dist[dist.index.year < 2026].nsmallest(5).round(1)

## Nivell 3 · Més enllà

### 3.1 · L'any típic

Calcula el percentatge mitjà de cada embassament per mes de l'any (de gener a desembre), amb totes les dades. En quin mes estan més plens? Tots igual?

In [ ]:
mes = df.pivot_table(index=df["dia"].dt.month, columns="estaci", values="percentatge_volum_embassat", aggfunc="mean")
print(mes.shape)
print(mes.idxmax().to_string())

### 3.2 · El que es va buidar més

Per a cada embassament, quin percentatge mínim va tenir i quin dia? I quan va tornar a superar el 50 %?

In [ ]:
minims = pd.DataFrame({"min": pct.min(), "dia": pct.idxmin()})
minims["torna_50"] = [pct.loc[pct.index > minims.loc[c, "dia"], c].gt(50).idxmax() if pct.loc[pct.index > minims.loc[c, "dia"], c].gt(50).any() else pd.NaT for c in pct.columns]
minims.sort_values("min")

## Conclusió de referència

La reserva total de les conques internes va tocar fons el 8 de març del 2024, amb 99,5 hm³: és el
mínim des del 2000, per sota del 2023 (116,5) i molt per sota de la sequera del 2008 (146,3). Tres
embassaments van quedar gairebé buits: Siurana (0,6 %), Sau (1,0 %) i Riudecanyes (1,6 %). Avui
(13-oct-2026) hi ha 507,1 hm³, i només 6 dels 26 anys anteriors en tenien més el mateix dia: la
sequera s'ha recuperat del tot. Per percentatges de cada embassament, els dies més semblants a avui
són de finals de juliol del 2015. Paranys: el percentatge del Foix supera el 100 % durant anys, i la
reserva del sistema s'ha de fer sumant hm³, no fent la mitjana dels percentatges.